# Anonymisation of the election survey


**Threat model.** The attacker has the public register (name, sex, date of birth, ZIP, citizenship, marital status, `last_voted`) and the official results. Every respondent is in the register with identical values and `evote` = `last_voted`, so those columns are a perfect linkage key. Education is **not** in the register and is assumed unknown to the attacker, so it is not a quasi-identifier (it is only released, never used for matching).

* **Outsider:** does not know who took the survey. Each released row is matched against all register voters. Risk of a row = 1/F, where F is the number of consistent voters.
* **Known participant** : knows that a specific person took the survey and looks up every released row consistent with that person's register data. The person is protected if there are **at least k = 5** such rows and they contain **both Red and Green** (l-diversity, l = 2).

A blank (NaN) cell is treated as "could be anything", which is how an attacker would read it.

In [1]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)

results = pd.read_excel("data/public_data_resultsM.xlsx").rename(columns={"Unnamed: 0": "unit"})
old = pd.read_csv("data/anonymised_dataM.csv")   # the original anonymised release


def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

## 1. Recoding (shared by the attack and the new release)

In [2]:
ANON_REF_DATE = pd.Timestamp("2025-07-01")
K, L = 5, 2                 # every known participant must match >= K released rows with >= L parties
SWAP_FRAC = 0.10            # share of rows whose party takes part in a swap
REGION = {2100: "2100-2200", 2200: "2100-2200", 2300: "2300-2400", 2400: "2300-2400"}
EDU_ANON = {
    "Primary education": "Primary",
    "Upper secondary education": "Secondary/VET", "Vocational Education and Training (VET)": "Secondary/VET",
    "Short cycle higher education": "Short/medium higher", "Vocational bachelors educations": "Short/medium higher",
    "Bachelors programmes": "Short/medium higher",
    "Masters programmes": "Long higher", "PhD programmes": "Long higher",
    "Not stated": "Not stated",
}
QI = ["sex", "age_group", "region", "marital_status"]          # known from the register (plus evote)
QI_COST = {"marital_status": 1, "region": 1, "sex": 1, "age_group": 3}  # age is kept unless unavoidable

priv = pd.read_excel("data/private_dataM.xlsx")
reg = pd.read_excel("data/public_data_registerM.xlsx")


def recode_qi(df, evote):
    out = pd.DataFrame(index=df.index)
    age = ((ANON_REF_DATE - df["dob"]).dt.days / 365.25).astype(int)
    out["sex"] = df["sex"]
    out["age_group"] = pd.cut(age, [0, 39, 59, 200], labels=["20-39", "40-59", "60+"]).astype(str)
    out["zip"] = df["zip"]
    out["region"] = df["zip"].map(REGION)
    out["marital_status"] = np.where(df["marital_status"] == "Married/separated", "Married/separated", "Not married")
    out["evote"] = evote
    return out


truth_q = recode_qi(priv, priv["evote"])                    # what an attacker knows about each participant
reg_voters_q = recode_qi(reg[reg["last_voted"].isin([0, 1])], reg["last_voted"])

anon = truth_q.drop(columns="zip").copy()
anon["education"] = priv["education"].map(EDU_ANON)
anon["party"] = priv["party"]
anon = anon[["sex", "age_group", "region", "education", "marital_status", "evote", "party"]]

## 2. Attack tools

In [3]:
def match_matrix(targets, released, qi):
    """M[i, j]: row j of `released` is consistent with row i of `targets` (a NaN cell matches anything)."""
    M = targets["evote"].values[:, None] == released["evote"].values[None, :]
    for c in qi:
        tv, rv = targets[c].values, released[c].values
        M &= (tv[:, None] == rv[None, :]) | pd.isna(tv)[:, None] | pd.isna(rv)[None, :]
    return M


def diversity(M, party):
    rg = np.isin(party, ["Red", "Green"])
    return np.array([len(set(party[m & rg])) for m in M])


def risk_report(released, qi, truth, register_q):
    rel_party = released["party"].values
    # outsider: every released row against all register voters
    F = match_matrix(released, register_q, qi).sum(axis=1)
    # known participant: every participant against all released rows
    M = match_matrix(truth, released, qi)
    n, div = M.sum(axis=1), diversity(M, rel_party)
    guess = np.array([pd.Series(rel_party[m]).mode()[0] for m in M])
    true = priv["party"].values
    return {
        "min F (register)": F.min(), "mean risk 1/F": (1 / F).mean(), "max risk 1/F": (1 / F).max(),
        f"participants matching < {K} rows": int((n < K).sum()),
        "participants with 1 party only": int((div < 2).sum()),
        "attacker accuracy (majority guess)": (guess == true).mean(),
    }

## 3. Attacking the original release
The age groups in the original release are reproduced exactly with a reference date around mid-2025, so an attacker can recompute them from the register.

In [4]:
old_risk = risk_report(old, ["sex", "age_group", "zip", "marital_status"], truth_q, reg_voters_q)
print("suppressed cells in the original release:", old.isna().sum().to_dict())
print("ZIP blanks that also blank marital status:", int((old["zip"].isna() & old["marital_status"].isna()).sum()),
      "of", int(old["zip"].isna().sum()))
print("rows still carrying a full 4-digit ZIP:", int(old["zip"].notna().sum()))
print("party counts original:", priv["party"].value_counts().to_dict(),
      "| released:", old["party"].value_counts().to_dict())
pd.Series(old_risk).round(4)

suppressed cells in the original release: {'sex': 3, 'age_group': 0, 'zip': 34, 'education': 0, 'marital_status': 63, 'evote': 0, 'party': 0}
ZIP blanks that also blank marital status: 31 of 34
rows still carrying a full 4-digit ZIP: 166
party counts original: {'Green': 127, 'Red': 71, 'Invalid vote': 2} | released: {'Green': 127, 'Red': 73}


min F (register)                       6.0000
mean risk 1/F                          0.0482
max risk 1/F                           0.1667
participants matching < 5 rows        48.0000
participants with 1 party only         9.0000
attacker accuracy (majority guess)     0.7000
dtype: float64

### Choosing how to code ZIP
Risk of the recoded data with each ZIP option, before swapping or suppression (lower is better):

In [5]:
# risk of each ZIP coding on the recoded data, before any swapping or suppression
base = truth_q.assign(party=priv["party"].values)
truth_z = truth_q.assign(zip_paper=truth_q["zip"])
reg_z = reg_voters_q.assign(zip_paper=reg_voters_q["zip"])
options = {
    "full ZIP": ("zip", base),
    "ZIP for paper voters only": ("zip_paper", base.assign(zip_paper=base["zip"].where(base["evote"] == 0))),
    "2 regions": ("region", base),
    "no ZIP": (None, base),
}
zip_risk = pd.DataFrame({
    name: risk_report(rel, ["sex", "age_group", "marital_status"] + ([col] if col else []), truth_z, reg_z)
    for name, (col, rel) in options.items()
}).T
zip_risk.round(4)

,min F (register),mean risk 1/F,max risk 1/F,participants matching < 5 rows,participants with 1 party only,attacker accuracy (majority guess)
full ZIP,3.0,0.0808,0.3333,117.0,76.0,0.780
ZIP for paper voters only,4.0,0.0539,0.2500,80.0,59.0,0.760
2 regions,7.0,0.0418,0.1429,51.0,47.0,0.745
no ZIP,17.0,0.0212,0.0588,14.0,31.0,0.730


## 4. Building the new release
1. *Swapping:* the party of 10 % of rows is permuted **within channel × region strata**, so vote counts per stratum (needed for question A) are unchanged. This happens before suppression, so the k/l checks apply to the values that are actually released.
2. *Local suppression:* greedy. For each participant who violates k-anonymity or l-diversity, the cheapest other row (same channel, and for l-diversity a different party) is pulled into that person's match set by blanking the cells where it differs. Marital status, region and sex cost 1, age costs 3, and `evote` is never blanked. Blanking only adds matches, so fixing one person never breaks another and the loop always ends.
3. Rows are shuffled and written to `data/anonymised_dataM_v2.csv`.

In [6]:
rng_anon = np.random.default_rng(2025)

# 1) swap party within (channel, region) strata: keeps the per-stratum vote counts, so the
#    comparison with the official results is unchanged
for _, idx in anon.groupby(["evote", "region"]).groups.items():
    chosen = rng_anon.choice(idx, size=int(round(SWAP_FRAC * len(idx))), replace=False)
    anon.loc[chosen, "party"] = rng_anon.permutation(anon.loc[chosen, "party"].values)


# 2) greedy local suppression until every participant is k-anonymous and l-diverse (Red/Green).
#    Suppressing a cell only ever adds matches, so fixing one target never breaks another.
n_steps = 0
while True:
    M = match_matrix(truth_q, anon, QI)
    n, div = M.sum(axis=1), diversity(M, anon["party"].values)
    bad = np.where((n < K) | (div < L))[0]
    if len(bad) == 0:
        break
    i = bad[np.argmin(n[bad])]
    seen = set(anon["party"].values[M[i]])
    cand = (~M[i]) & (anon["evote"].values == truth_q["evote"].values[i])
    if div[i] < L:
        cand &= ~anon["party"].isin(seen | {"Invalid vote"}).values
    cand = np.where(cand)[0]
    # cost of pulling row j into target i's match set = cells that would have to be suppressed
    mism = {c: anon[c].notna().values[cand] & (anon[c].values[cand] != truth_q[c].values[i]) for c in QI}
    cost = sum(QI_COST[c] * mism[c] for c in QI) + 0.01 * anon.iloc[cand][QI].isna().sum(axis=1).values
    best = int(np.argmin(cost))
    j = cand[best]
    for c in QI:
        if mism[c][best]:
            anon.iat[j, anon.columns.get_loc(c)] = np.nan
    n_steps += 1

# 3) shuffle the row order so it carries no information
anon = anon.sample(frac=1, random_state=2025).reset_index(drop=True)
anon.to_csv("data/anonymised_dataM_v2.csv", index=False, na_rep="NA")  # same blank marker as the original release
print(f"{n_steps} suppression steps; suppressed cells per column:")
print(anon.isna().sum().to_string())

66 suppression steps; suppressed cells per column:
sex               18
age_group          0
region            33
education          0
marital_status    15
evote              0
party              0


## 5. Risk: original vs new release

In [7]:
risk = pd.DataFrame({
    "previous release": risk_report(old, ["sex", "age_group", "zip", "marital_status"], truth_q, reg_voters_q),
    "new release": risk_report(anon, QI, truth_q, reg_voters_q),
}).T
print(f"baseline accuracy of always guessing Green: {(priv['party'] == 'Green').mean():.3f}")
risk.round(4)

baseline accuracy of always guessing Green: 0.635


,min F (register),mean risk 1/F,max risk 1/F,participants matching < 5 rows,participants with 1 party only,attacker accuracy (majority guess)
previous release,6.0,0.0482,0.1667,48.0,9.0,0.700
new release,8.0,0.0330,0.1250,0.0,0.0,0.735


## 6. Utility: are the analysis results preserved?
Cramér's V and χ² p-values for the associations tested in questions (B) and (C), on the original data (same recoding, no suppression or swapping) vs the released data (blank cells dropped).

In [8]:
orig = truth_q.drop(columns="zip").assign(education=priv["education"].map(EDU_ANON), party=priv["party"])
rows = []
for outcome, attrs in [("party", ["sex", "age_group", "education", "marital_status", "region", "evote"]),
                       ("evote", ["sex", "age_group", "education", "marital_status", "region"])]:
    for a_ in attrs:
        r_ = {"outcome": outcome, "attribute": a_}
        for label, df in [("original", orig), ("released", anon)]:
            d = df[[a_, outcome]].dropna()
            if outcome == "party":
                d = d[d["party"].isin(["Red", "Green"])]
            ct = pd.crosstab(d[a_], d[outcome])
            r_[f"V {label}"] = cramers_v(ct)
            r_[f"p {label}"] = stats.chi2_contingency(ct, correction=False)[1]
        rows.append(r_)
utility = pd.DataFrame(rows).set_index(["outcome", "attribute"])
utility.round(4)

V original  p original  V released  p released
outcome attribute                                                     
party   sex                 0.0829      0.2436      0.0463      0.5337
        age_group           0.3688      0.0000      0.3436      0.0000
        education           0.3679      0.0000      0.3217      0.0004
        marital_status      0.0575      0.4188      0.0355      0.6307
        region              0.2084      0.0034      0.1555      0.0457
        evote               0.1114      0.1168      0.1114      0.1168
evote   sex                 0.0043      0.9514      0.0070      0.9250
        age_group           0.2788      0.0004      0.2788      0.0004
        education           0.2037      0.0813      0.2037      0.0813
        marital_status      0.0419      0.5533      0.0454      0.5370
        region              0.0408      0.5637      0.0029      0.9700

Per-ZIP comparisons for question (A) are no longer possible, so the survey is compared with the official results per region instead:

In [ ]:
# per-stratum vote counts survive the swap, so question (A) can still be answered per region
off = results.set_index("unit")
off_region = {}
for z, rg in REGION.items():
    off_region.setdefault(rg, 0)
    off_region[rg] = off_region[rg] + off.loc[f"Polling station: ZIP {z}", ["Red", "Green"]].astype(float)
rows = []
for rg, o in off_region.items():
    for label, df, col in [("original", priv.assign(region=priv["zip"].map(REGION)), "region"),
                           ("released", anon, "region")]:
        sub = df[(df["evote"] == 0) & (df[col] == rg)]
        rows.append({"region": rg, "data": label, "n": len(sub), "Red": (sub["party"] == "Red").sum(),
                     "Green": (sub["party"] == "Green").sum(), "official Red share": o["Red"] / o.sum()})
reg_cmp = pd.DataFrame(rows)
reg_cmp["survey Red share"] = reg_cmp["Red"] / (reg_cmp["Red"] + reg_cmp["Green"])
reg_cmp.round(3)

,region,data,n,Red,Green,official Red share,survey Red share
0,2100-2200,original,59,15,43,0.280,0.259
1,2100-2200,released,53,15,37,0.280,0.288
2,2300-2400,original,77,38,38,0.396,0.500
3,2300-2400,released,68,31,36,0.396,0.463
